# MrChicken + Easy-Wav2Lip (com GFPGAN) no Kaggle

Este notebook executa o Easy-Wav2Lip (sincronização labial de alta qualidade com restauração facial GFPGAN) como um microserviço REST, compatível com o MrChicken local.

In [ ]:
from pathlib import Path
import os, secrets, subprocess, sys, time

WORK_DIR = Path('/kaggle/working')
REPO_DIR = WORK_DIR / 'Easy-Wav2Lip'
SERVICE_FILE = WORK_DIR / 'mrchicken_lipsync_service.py'
OUTPUTS_DIR = WORK_DIR / 'mrchicken_lipsync_outputs'
PORT = int(os.environ.get('LIPSYNC_PORT', '8010'))

API_KEY = ""
os.environ['LIPSYNC_API_KEY'] = API_KEY
os.environ['MUSETALK_OUTPUTS_DIR'] = str(OUTPUTS_DIR)
os.environ['MUSETALK_TIMEOUT_SECONDS'] = '1800'
os.environ['HF_HOME'] = str(WORK_DIR / 'hf-cache')

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_DIR=', WORK_DIR)
print('REPO_DIR=', REPO_DIR)
print('OUTPUTS_DIR=', OUTPUTS_DIR)
print('PORT=', PORT)


In [ ]:
# Clone/update do Easy-Wav2Lip
import shutil, subprocess, socket, os

giturl = 'https://github.com/anothermartz/Easy-Wav2Lip.git'
version = 'v8.3'

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', version, giturl, str(REPO_DIR)], check=True)

# Instala pré-requisitos compatíveis com Python 3.12 no Kaggle
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'batch_face', 'basicsr==1.4.2', 'gfpgan', 'fastapi', 'uvicorn', 'python-multipart', '--quiet'], check=True)

# Corrigir bug clássico do basicsr degradations.py com Python 3.10+
# Corrigir bug clássico do basicsr degradations.py com Python 3.10+ sem importar basicsr
import site
basicsr_path = None
for d in site.getsitepackages():
    candidate = Path(d) / 'basicsr'
    if candidate.exists():
        basicsr_path = candidate
        break
if basicsr_path:
    basicsr_data_path = basicsr_path / 'data' / 'degradations.py'
    source_degradations = REPO_DIR / 'degradations.py'
    if source_degradations.exists() and basicsr_data_path.exists():
        shutil.copy2(source_degradations, basicsr_data_path)
        print("basicsr degradations.py corrigido com sucesso!")
else:
    print("Pasta basicsr não encontrada no site-packages.")

# Criar pastas internas
(REPO_DIR / 'face_alignment').mkdir(parents=True, exist_ok=True)
(REPO_DIR / 'temp').mkdir(parents=True, exist_ok=True)

# Rodar instalador oficial para baixar pesos do Easy-Wav2Lip
print("Iniciando install.py oficial para baixar pesos...")
subprocess.run([sys.executable, 'install.py'], cwd=str(REPO_DIR), check=True)
print("Instalação concluída com sucesso!")


In [ ]:
# Download de pesos com links diretos caso o install.py falhe ou seja lento
import subprocess
weights = {
    "checkpoints/wav2lip_gan.pth": "https://github.com/anothermartz/Easy-Wav2Lip/releases/download/Prerequesits/Wav2Lip_GAN.pth",
    "weights/GFPGANv1.4.pth": "https://github.com/xinntao/Face-Restoration-TencentPlayground/releases/download/v0.1.0/GFPGANv1.4.pth"
}

for rel_path, url in weights.items():
    dest = REPO_DIR / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists() or dest.stat().st_size < 1000000:
        print(f"Baixando {rel_path}...")
        try:
            subprocess.run(['wget', '-q', url, '-O', str(dest)], check=True)
            print(f"{rel_path} baixado com sucesso!")
        except Exception as e:
            print(f"Erro ao baixar {rel_path}: {e}")


In [ ]:
SERVICE_SOURCE = r'''
import os
import re
import shutil
import subprocess
import time
from pathlib import Path
from typing import Optional
from fastapi import FastAPI, File, Form, Header, HTTPException, Request, UploadFile
from fastapi.responses import FileResponse
from pydantic import BaseModel, ConfigDict, Field
import configparser

app = FastAPI(title="MrChicken Easy-Wav2Lip Service", version="1.0.0")

WORK_DIR = Path('/kaggle/working')
REPO_DIR = WORK_DIR / 'Easy-Wav2Lip'
OUTPUTS_DIR = WORK_DIR / 'mrchicken_lipsync_outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

class GenerateResponse(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    success: bool
    video_path: str = Field(..., alias='videoPath')
    video_url: Optional[str] = Field(default=None, alias='videoUrl')

def verify_api_key(authorization: Optional[str] = Header(default=None), x_api_key: Optional[str] = Header(default=None)) -> None:
    return  # Keyless

def safe_path_segment(value: str) -> str:
    return re.sub(r'[^a-zA-Z0-9._-]', '-', value)[:120] or 'job'

def safe_upload_name(filename: str | None, fallback: str) -> str:
    suffix = Path(filename or fallback).suffix.lower()
    if suffix not in {'.jpg', '.jpeg', '.png', '.webp', '.mp4', '.mov', '.webm', '.wav', '.mp3', '.m4a'}:
        suffix = Path(fallback).suffix
    return f'{Path(fallback).stem}{suffix}'

def save_upload(upload: UploadFile, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('wb') as output:
        shutil.copyfileobj(upload.file, output)
    return destination

def run_easy_wav2lip(job_id: str, avatar_path: Path, audio_path: Path) -> Path:
    job_dir = OUTPUTS_DIR / safe_path_segment(job_id)
    job_dir.mkdir(parents=True, exist_ok=True)
    output_path = job_dir / "wav2lip-output.mp4"
    
    # Escreve config.ini para o Easy-Wav2Lip
    config = configparser.ConfigParser()
    config['OPTIONS'] = {
        'video_file': str(avatar_path),
        'vocal_file': str(audio_path),
        'quality': 'Improved',  # Wav2Lip + Mask + GFPGAN (Restauração facial)
        'output_height': 'full resolution',
        'wav2lip_version': 'Wav2Lip_GAN',
        'use_previous_tracking_data': 'True',
        'nosmooth': 'True'
    }
    config['PADDING'] = {
        'U': '0',
        'D': '10',
        'L': '0',
        'R': '0'
    }
    config['MASK'] = {
        'size': '1.5',
        'feathering': '1',
        'mouth_tracking': 'False',
        'debug_mask': 'False'
    }
    config['OTHER'] = {
        'batch_process': 'False',
        'output_suffix': '_Easy-Wav2Lip',
        'include_settings_in_suffix': 'False',
        'preview_input': 'False',
        'preview_settings': 'False',
        'frame_to_preview': '100'
    }
    
    config_file = REPO_DIR / 'config.ini'
    with open(config_file, 'w') as f:
        config.write(f)
        
    cmd = [os.environ.get('PYTHON') or 'python', 'run.py']
    print(f"Executando Easy-Wav2Lip: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True, check=False)
    if result.stdout:
        print(result.stdout[-2000:])
    if result.stderr:
        print(result.stderr[-2000:])
        
    if result.returncode != 0:
        raise RuntimeError(f"Easy-Wav2Lip falhou com código {result.returncode}")
        
    # O output gerado é salvo na pasta temp/output.mp4
    temp_output = REPO_DIR / 'temp' / 'output.mp4'
    if not temp_output.exists():
        raise FileNotFoundError("Vídeo resultante não encontrado em temp/output.mp4")
        
    shutil.copy2(temp_output, output_path)
    return output_path

@app.get('/health')
def health() -> dict[str, object]:
    import torch
    return {'success': True, 'engine': 'easy-wav2lip', 'gpuAvailable': torch.cuda.is_available()}

@app.post('/generate-upload')
async def generate_upload(request: Request, jobId: str = Form(..., min_length=1), avatar: UploadFile = File(...), audio: UploadFile = File(...)):
    safe_job_id = safe_path_segment(jobId)
    job_dir = OUTPUTS_DIR / safe_job_id
    avatar_path = save_upload(avatar, job_dir / safe_upload_name(avatar.filename, 'avatar.jpg'))
    audio_path = save_upload(audio, job_dir / safe_upload_name(audio.filename, 'audio.wav'))
    import asyncio
    import json
    from fastapi.responses import StreamingResponse
    async def event_generator():
        loop = asyncio.get_running_loop()
        task = loop.run_in_executor(None, run_easy_wav2lip, safe_job_id, avatar_path, audio_path)
        while not task.done():
            yield ' '
            await asyncio.sleep(5)
        try:
            video_path = await task
            result = {
                'success': True,
                'videoPath': str(video_path),
                'videoUrl': str(request.url_for('download_output', job_id=safe_job_id, filename=video_path.name))
            }
        except Exception as exc:
            import traceback
            error_text = traceback.format_exc()
            (job_dir / 'error.log').write_text(error_text, encoding='utf-8')
            result = {'success': False, 'error': str(exc), 'code': 'WAV2LIP_ERROR'}
        yield json.dumps(result)
    return StreamingResponse(event_generator(), media_type='application/json')

@app.get('/outputs/{job_id}/{filename}', name='download_output')
def download_output(job_id: str, filename: str) -> FileResponse:
    output_path = OUTPUTS_DIR / safe_path_segment(job_id) / Path(filename).name
    if not output_path.exists():
        raise HTTPException(status_code=404, detail={'success': False, 'error': f'Output não encontrado: {filename}', 'code': 'FILE_NOT_FOUND'})
    return FileResponse(output_path, media_type='video/mp4', filename=Path(filename).name)
'''
SERVICE_FILE.write_text(SERVICE_SOURCE, encoding='utf-8')
print('Serviço escrito em', SERVICE_FILE)


In [ ]:
# Inicia FastAPI localmente e expõe com Cloudflare Tunnel temporário.
import os, queue, re, stat, threading, subprocess, time, requests, sys
os.environ['PYTHON'] = sys.executable

cloudflared = WORK_DIR / 'cloudflared'
if not cloudflared.exists():
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', str(cloudflared)], check=True)
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

for name in ['server_proc', 'tunnel_proc']:
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()

server_proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'mrchicken_lipsync_service:app', '--host', '0.0.0.0', '--port', str(PORT), '--proxy-headers'], cwd=str(WORK_DIR), env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

def stream_logs(prefix, proc):
    for line in proc.stdout:
        print(prefix, line, end='')
threading.Thread(target=stream_logs, args=('[uvicorn]', server_proc), daemon=True).start()
time.sleep(5)
resp = requests.get(f'http://127.0.0.1:{PORT}/health', timeout=15)
print('Health local:', resp.status_code, resp.text[:500])

tunnel_proc = subprocess.Popen([str(cloudflared), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
q = queue.Queue()
def capture_tunnel():
    for line in tunnel_proc.stdout:
        print('[cloudflared]', line, end='')
        q.put(line)
threading.Thread(target=capture_tunnel, daemon=True).start()

PUBLIC_URL = None
pattern = re.compile(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com')
deadline = time.time() + 90
while time.time() < deadline and PUBLIC_URL is None:
    try:
        line = q.get(timeout=2)
    except queue.Empty:
        continue
    match = pattern.search(line)
    if match:
        PUBLIC_URL = match.group(0)
if not PUBLIC_URL:
    raise RuntimeError('Não consegui obter URL pública do cloudflared.')

print('\n=== CONFIGURE NO .env.local DO MRCHICKEN ===')
print(f'LIPSYNC_API_URL={PUBLIC_URL}')
print('# LIPSYNC_API_KEY não é necessária')
print('LIPSYNC_TRANSFER_MODE=upload')
print('LIPSYNC_TIMEOUT_MS=1800000')
print('Endpoint público:', PUBLIC_URL)
